# What a CRISM band is

A CRISM observation arrives as a cube of numbers with no wavelengths anywhere in it. The
label says the file holds 55 bands and says nothing about what any of those 55 bands
looked at. Band 0 of one observation and band 0 of the next need not be the same colour
of light.

This notebook opens one observation and prints, as plain tables, the three things that
have to be lined up:

- **The TRDR as it is written**, which is what the instrument sent down.
- **The CDR**, which is the ground record saying what each band was looking at.
- **The TRDR after calibration**, which is the two of them put together.

Every number below is read with `preprocessing.crism`, the pipeline's own reader. Nothing
is recomputed here for show.

## How the numbers are made

CRISM looks at Mars through a narrow slit. The slit sees one thin strip of ground lying
across the direction of flight. The light from that strip is spread out by a grating, so
that colour runs one way and position along the slit runs the other, and it falls on a
rectangular detector. One read of that detector is one frame: across it, every column is
a place on the ground, and every row is a colour.

The spacecraft moves forward and the next frame is the next strip. Stacking the frames in
the order they were taken builds the image. So the three axes of the file are

- **lines**, which is time, and therefore distance along the ground track,
- **samples**, which is position across the slit,
- **bands**, which is one row of the detector.

The third axis is the whole problem. A band is a detector row, not a wavelength. Which
colour a row sees is a fact about the optics: where the grating throws each colour and how
the slit is imaged onto the glass. It was measured on the ground before launch, and it is
not written in the observation.

Survey mode sharpens the problem, twice over, and both times to fit the downlink. Ten
detector columns are averaged into one sample, so 640 columns arrive as 64. And only a few
dozen of the detector's rows are sent down at all, picked by a table uploaded to the
instrument. Which rows are picked is not the same for every observation.

There are two detectors, and they are read as two separate files. `S` covers the visible
and `L` covers the infrared, so one scan is always two cubes with different band counts.

## Setup

In [ ]:
"""Bring one observation down and name what to look at inside it."""

from pathlib import Path

import numpy as np

from preprocessing.crism.calibration import bands_calibration, wavelengths
from preprocessing.crism.fetching import download
from preprocessing.crism.storage import locations, naming, reading

OBSERVATION = "msp00006994_05_if214_trr3"

LINE, SAMPLE = 1000, 32

CDR_ROOT = Path(wavelengths.__file__).parent / "cdr"

for detector, label in download.fetch(OBSERVATION).items():
    print(detector, label.name)

In [ ]:
"""Two printers, so every table below is laid out the same way."""


def edges(size, keep=3):
    """Return the first and last few indices of an axis.

    Args:
        size: How long the axis is.
        keep: How many indices to take from each end.

    Returns:
        The indices, first end then last.
    """
    return list(range(keep)) + list(range(size - keep, size))


def block(grid, rows, cols, digits=2):
    """Print part of a two dimensional grid as a plain table.

    Args:
        grid: The grid to read from.
        rows: Which row indices to print, in the order to print them.
        cols: Which column indices to print, in the order to print them.
        digits: How many decimals to show each value with.

    Returns:
        None.
    """
    print("      " + "".join(f"{col:>10}" for col in cols))
    for row in rows:
        cells = "".join(f"{grid[row, col]:>10.{digits}f}" for col in cols)
        print(f"{row:>6}" + cells)

## The TRDR as it comes off disk

The label is the only description the file carries of its own shape. It gives the three
axis lengths, the order the values were written in, and the width of one number. It does
not give a single wavelength.

`PIXEL_AVERAGING_WIDTH` is the binning: 10 detector columns per sample, which is where 64
samples comes from.

In [ ]:
"""What each detector's label says about the shape of its cube."""

head = f"{'det':>4}{'lines':>8}{'samples':>9}{'bands':>7}{'bin':>5}"
print(f"{head}  {'order':<19}unit")
for name in naming.DETECTORS:
    image = locations.files(OBSERVATION, name)[".img"]
    label = reading.load_label(image.with_suffix(".lbl"))
    lines, samples, bands, stored, dtype = reading.load_layout(label)
    row = f"{name:>4}{lines:>8}{samples:>9}{bands:>7}"
    print(f"{row}{label['PIXEL_AVERAGING_WIDTH']:>5}  {stored:<19}{label['UNIT']}")

### One band of the raw cube

This is a slice through the infrared cube at a single band: the rows are lines down the
track, the columns are samples across the slit. The values are I/F, the fraction of the
sunlight arriving that came back up.

The first three columns and the last one are not measurements. The file writes `65535.0`
there, which is a flag meaning the detector was never calibrated at that column. Left as a
number it is roughly two hundred thousand times a real value, so any average that includes
it is ruined.

In [ ]:
"""The infrared cube exactly as the file writes it, at one band."""

image = locations.files(OBSERVATION, "l")[".img"]
label = reading.load_label(image.with_suffix(".lbl"))
raw = reading.build_cube(image, label)

print(f"cube {raw.shape}, band 10, lines down, samples across")
block(raw[:, :, 10], edges(raw.shape[0]), edges(raw.shape[1]), digits=3)

### One pixel of the raw cube

Fixing a line and a sample and reading across the bands gives a spectrum. This is what the
file offers: a value per band index, and no way to know what band index means.

Band 0 is `65535.0` down the whole cube. That row of the detector was downlinked but never
calibrated, so the entire band is a flag.

In [ ]:
"""The spectrum the file gives at one pixel, indexed only by band number."""

print(f"line {LINE}, sample {SAMPLE}, {raw.shape[2]} bands")
print("".join(f"{band:>10}" for band in edges(raw.shape[2], 5)))
print("".join(f"{raw[LINE, SAMPLE, band]:>10.3f}" for band in edges(raw.shape[2], 5)))

## What the file does know: detector rows

The image is not the only thing in the `.img`. A short binary table is appended after it,
listing the detector row each band was read from. The pipeline's reader skips it on
purpose, by reading exactly as many values as the label's shape accounts for, but it is
worth looking at once because it is the missing link.

The numbers ascend and are not evenly spaced. They are rows on a piece of glass. They are
still not wavelengths, but they are the key that the ground record is written against.

In [ ]:
"""The detector rows each band was read from, appended after the image."""

for name in naming.DETECTORS:
    path = locations.files(OBSERVATION, name)[".img"]
    shape = reading.load_layout(reading.load_label(path.with_suffix(".lbl")))
    bands = shape[2]
    offset = shape[0] * shape[1] * bands * 4
    rows = np.fromfile(path, dtype=">u2", count=bands, offset=offset) & 0x1FF
    print(f"{name}, {bands} bands, detector rows:")
    print(rows)

## The CDR, which says what each row saw

A CDR is a Calibration Data Record: a file published once, on the ground, that describes
the instrument rather than any one observation. There are many kinds. The one that matters
here is `WA`, the wavelength record, and it is built by pointing the instrument at light of
known colour and writing down where on the detector each colour landed.

The result is a table with one number per detector column and per band: the centre
wavelength, in nanometres, that this column of this row is looking at.

It is a table and not a list because wavelength depends on the column as well as the row.
The slit is imaged onto the detector as a slight curve, not a straight line, so a single
detector row drifts in colour from one edge of the swath to the other. That curve is called
spectral smile, and it is measured further down.

Every observation's label names the record it was calibrated against, and the label is the
one place the choice is written.

In [ ]:
"""Which wavelength record each detector of this observation names."""

for name in naming.DETECTORS:
    label = reading.load_label(locations.files(OBSERVATION, name)[".lbl"])
    print(f"{name}  bands {label['BANDS']:>3}  {label['MRO:WAVELENGTH_FILE_NAME']}")

### The records the repository keeps

Survey mode was only ever seen naming two configurations per detector, a coarser downlink
and a finer one, so only those are kept. They span the same wavelengths at different
sampling. The visible detector has two band counts per record because some observations
drop its first band, which is why one record serves both.

`wavelengths.load` picks the record from the detector and the band count, so the band count
is doing the work the label's record name would do.

In [ ]:
"""What each kept record covers, read through the loader."""

head = f"{'det':>4}{'bands':>7}{'cols':>6}{'live':>6}"
print(f"{head}  {'range (nm)':<20}{'step':>8}{'smile':>8}")
for name, bands in (("l", 55), ("l", 70), ("s", 18), ("s", 19), ("s", 24), ("s", 25)):
    table = wavelengths.load(name, bands)
    centres = bands_calibration.centres(table)
    live = int((~np.isnan(table).all(axis=1)).sum())
    named = table[:, ~np.isnan(table).all(axis=0)]
    smile = (np.nanmax(named, axis=0) - np.nanmin(named, axis=0)).max()
    ends = centres[~np.isnan(centres)]
    span = f"{ends[0]:.1f} to {ends[-1]:.1f}"
    step = np.abs(np.diff(ends)).mean()
    row = f"{name:>4}{bands:>7}{table.shape[0]:>6}{live:>6}"
    print(f"{row}  {span:<20}{step:>8.1f}{smile:>8.1f}")

### The record as bytes, then as the loader reads it

The `.img` of a CDR is built the same way an observation is: a plain grid of 32 bit floats
with a label beside it. It holds one line, 64 columns and one value per band, and it writes
the same `65535` flag where nothing was ever calibrated.

The first table is the file untouched. The second is what `wavelengths.load` returns, which
is the same grid with the flag turned into `NaN` so that averaging it is impossible rather
than merely wrong.

In [ ]:
"""The infrared record at 55 bands, as written and as loaded."""

stored = np.fromfile(CDR_ROOT / "infrared_55.img", dtype="<f4", count=64 * 55)
stored = stored.reshape(55, 64).T
loaded = wavelengths.load("l", 55)

print("as written, columns down, bands across")
block(stored, edges(64), edges(55), digits=1)
print()
print("as loaded")
block(loaded, edges(64), edges(55), digits=1)

Reading down a column of that table shows the wavelengths falling as the band index rises.
The infrared record is written long wavelength first. The visible record is written the
other way round, short wavelength first. Neither file says which it is, so the direction
has to be read off the numbers.

Reading across a row shows the smile: the same band, drifting in colour from one side of
the swath to the other.

In [ ]:
"""How far one band drifts in wavelength across the swath."""

for name, bands in (("l", 55), ("s", 19)):
    table = wavelengths.load(name, bands)
    print(f"{name}, {bands} bands")
    print(f"{'band':>6}{'min (nm)':>12}{'max (nm)':>12}{'spread':>10}")
    for band in edges(bands):
        column = table[:, band]
        if np.isnan(column).all():
            print(f"{band:>6}{'never calibrated':>34}")
            continue
        low, high = np.nanmin(column), np.nanmax(column)
        print(f"{band:>6}{low:>12.2f}{high:>12.2f}{high - low:>10.2f}")
    print()

## Putting the two together

`bands_calibration.calibrate` does three things and nothing else. It reads the record for
this detector at this band count. It reverses the cube and the record together when the
record runs long wavelength first, so that every cube leaves in the same direction. And it
writes `NaN` over the columns and the bands the record never calibrated, so the flag never
survives as a number.

The shape does not change. A dead band stays a band, it just stops pretending to be data.

In [ ]:
"""The same pixel before and after, and what each band is centred on."""

ordered, table = bands_calibration.calibrate(raw, "l")
centres = bands_calibration.centres(table)

print(f"{'band':>6}{'stored':>12}{'wavelength':>13}{'calibrated':>13}")
for band in edges(ordered.shape[2], 5):
    print(
        f"{band:>6}{raw[LINE, SAMPLE, band]:>12.3f}"
        f"{centres[band]:>13.1f}{ordered[LINE, SAMPLE, band]:>13.3f}"
    )

The `stored` column and the `calibrated` column are not two versions of the same number.
They are two different bands that happen to share an index, because the reversal moved band
54 to position 0. The wavelength column is the only one that means anything on its own.

The same slice of the cube printed earlier now has `NaN` at the columns that were `65535`.

In [ ]:
"""The cube after calibration, at the band the raw slice was printed at."""

print(f"cube {ordered.shape}, band 44, lines down, samples across")
block(ordered[:, :, 44], edges(ordered.shape[0]), edges(ordered.shape[1]), digits=3)

## Checking that nothing was invented

Three things are worth confirming rather than trusting.

The flag and the record agree: every `65535` the observation wrote is at a column or band
the record refuses to name, and there are no others. If the two disagreed, either real data
would be thrown away or a flag would survive as a number.

The reader gives the same cube: `reading.read` runs the calibration itself, so opening the
observation the ordinary way must land on exactly the array built by hand above.

The record is written against the rows the observation names: the wavelength record carries
its own trailing table of detector rows, and it is the same list the observation appended.
That is what makes band index a legitimate key between the two files.

In [ ]:
"""The flag in the observation covers exactly what the record leaves unnamed."""

record = wavelengths.load("l", raw.shape[2])

dead = np.zeros(raw.shape[1:], dtype=bool)
dead[np.isnan(record).all(axis=1), :] = True
dead[:, np.isnan(record).all(axis=0)] = True

flagged = (raw == wavelengths.UNCALIBRATED).all(axis=0)
print(f"columns and bands the record never names : {int(dead.sum())}")
print(f"columns and bands the file flags 65535   : {int(flagged.sum())}")
print(f"the same ones                            : {bool((dead == flagged).all())}")
print(f"values lost as a share of the cube       : {dead.mean():.2%}")

In [ ]:
"""Opening the observation the ordinary way gives the same array."""

observation = reading.read(OBSERVATION)
infrared = observation.infrared

same_cube = np.array_equal(infrared.cube, ordered, equal_nan=True)
same_waves = np.array_equal(infrared.wavelengths, table, equal_nan=True)

print(f"same values     : {same_cube}")
print(f"same wavelengths: {same_waves}")
print(f"visible cube    : {observation.visible.cube.shape}")
print(f"geometry cube   : {infrared.geometry.shape}")

In [ ]:
"""The record and the observation name the same detector rows."""

for name, bands, record in (("l", 55, "infrared_55.img"), ("s", 19, "visible_19.img")):
    path = locations.files(OBSERVATION, name)[".img"]
    shape = reading.load_layout(reading.load_label(path.with_suffix(".lbl")))
    after = shape[0] * shape[1] * bands * 4
    sent = np.fromfile(path, dtype=">u2", count=bands, offset=after)
    known = np.fromfile(
        CDR_ROOT / record, dtype=">u2", count=bands, offset=64 * bands * 4
    )
    print(f"{name}  rows match: {bool(((sent & 0x1FF) == (known & 0x1FF)).all())}")

## What this leaves the rest of the pipeline

A calibrated cube is still lines by samples by bands, and a band is still an index. What
has changed is that the index now means the same thing in every observation of a given
configuration: the bands ascend in wavelength, and the wavelengths themselves are carried
beside the cube rather than assumed.

Two things are deliberately not done here and are worth stating.

**Nothing is resampled.** The wavelength of a band still depends on the sample it is read
at, because the smile is real and is up to 16 nm on the infrared detector. Comparing one
sample against another at a fixed band index compares slightly different colours. The
wavelengths are kept as a full table for exactly that reason, so any later step can see the
drift instead of averaging it away.

**Dead columns and bands are kept in place.** They are `NaN`, not removed. Dropping them
would change the shape of the cube per configuration and break the correspondence with the
geometry backplanes, which are on the same grid.

The remaining assumption is the record choice. `wavelengths.load` selects on detector and
band count, while the label names the record outright. That holds because survey mode has
only ever been seen naming the two configurations kept here, and it is checked above by
printing what each label names.